## Load Packages

In [1]:
import pandas as pd 
import pyam
import matplotlib.pyplot as plt
import pandas_indexing as pix
from pandas_indexing import isin, ismatch
import numpy as np

import os
from pathlib import Path

from fuzzywuzzy import fuzz

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import data_shepherd as ds
from data_shepherd import utils
import datatoolbox as dt

<IPython.core.display.Javascript object>

In [2]:
def elec(tech=None):
    if not tech is None:
        return f'Secondary Energy|Electricity|{tech}'
    else:
        return 'Secondary Energy|Electricity'

In [3]:
REMIND_RMAP = ds.utils.RegionMapping.from_model('REMIND_2.1')

In [4]:
IKEA_ISOS = ["DEU","POL","BRA","MEX","KEN","MAR","MOZ",
             "NGA","SEN","ZAF","USA","NAM","DZA","TUR","SAU",
             "ARE","BGD","IND","IDN","PAK","VNM","GBR","AUS","CHN","JPN"]

LAM_ISOS = [
    'ABW', 'ARG', 'ATG', 'BHS', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'CHL', 'COL',
    'CRI', 'CUB', 'CYM', 'DMA', 'DOM', 'ECU', 'FLK', 'GLP', 'GRD', 'GTM', 'GUF',
    'GUY', 'HND', 'HTI', 'JAM', 'KNA', 'LCA', 'MEX', 'MSR', 'MTQ','NIC', 'PAN', 'PER',
    'PRI', 'PRY', 'SLV', 'SUR', 'TCA', 'TTO', 'URY', 'VCT', 'VEN', 'VGB', 'VIR']

IKEA_ISOS.sort()

## Load data

In [5]:
BOX_MOUNT_PATH = Path("~/Library/CloudStorage/Box-Box").expanduser()
if not BOX_MOUNT_PATH.is_dir():
    BOX_MOUNT_PATH = Path("~/Box").expanduser()

In [6]:
DSCALE_PATH = (
    "Climate Policy Team/"
    "02 - Projects/"
    "IKEA NDC 1.5° Pathways 23-25 - phase II/"
    "2 - Work Packages/"
    "WP2 - National 1.5°C Pathways/"
    "Downscaling/"
    "DSCALE"
    
)

DSCALE_PATH: os.PathLike = BOX_MOUNT_PATH / DSCALE_PATH

In [7]:
REMIND_PATH = (
    "Climate Policy Team/"
    "02 - Projects/"
    "IKEA NDC 1.5° Pathways 23-25 - phase II/"
    "2 - Work Packages/"
    "WP1 - Updating global 1.5°C Pathways/"
    "1 - PIK/"
    "Scenario Data/"
    "SUBMITTED_SCENARIOS_August2025/"

)

REMIND_PATH: os.PathLike = BOX_MOUNT_PATH / REMIND_PATH

In [8]:
# INSERT PATHWAY TO INPUT DATA
input_data = pyam.IamDataFrame('../input_data/REMIND_fuel_mix_testing/snapshot_v1/npe_core_scenario_harmonised_AR6_reporting-DAC.csv')
# input_data_processed = pyam.IamDataFrame('../input_data/REMIND_fuel_mix_testing/multiple_df/snapshot_all_regions_RAW_REMIND-MAgPIE 3.3-4.8.csv')

pyam - INFO: Running in a notebook, setting up a basic logging at level INFO
pyam.core - INFO: Reading file ../input_data/REMIND_fuel_mix_testing/snapshot_v1/npe_core_scenario_harmonised_AR6_reporting-DAC.csv


In [9]:
import pandas as pd

# INSERT PATHWAY TO HISTORICAL DATA HERE
df = pd.read_csv(
    DSCALE_PATH / 'data/step1_input_data_for_DSCALE/input_reference_iea_2022.csv'
)

iamc_cols = ["MODEL", "SCENARIO", "REGION", "VARIABLE", "UNIT"]

df_clean = df.drop_duplicates(
    subset=["MODEL", "SCENARIO", "REGION", "VARIABLE", "UNIT"]
)

hist_data = pyam.IamDataFrame(df_clean)

In [147]:
# PUT PATH TO YOUR DSCALE RESULTS IN HERE
dscale_results = pyam.IamDataFrame('../results/5_Explorer_and_New_Variables/REMIND 3.4_REMIND_fuel_mix_testing_04_02_2026_2022_harmo_step5g_synfuel.csv')

pyam.core - INFO: Reading file ../results/5_Explorer_and_New_Variables/REMIND 3.4_REMIND_fuel_mix_testing_04_02_2026_2022_harmo_step5g_synfuel.csv


In [148]:

# Regroup all the rescaled countries within their regions

region_map = {
    'R10AFRICA':'SSA',
    'R10CHINA+':'CHA',
    'R10EUROPE':'EUR',
    'R10LATIN_AM':'LAM',
    'R10MIDDLE_EAST':'MEA',
    'R10REF_ECON':'REF',
    'R10REST_ASIA':'OAS',
    'R10NORTH_AM':'USA',
    'R10PAC_OECD':'JPN',
    'R10INDIA+':'IND',
    'R10ROWO':'ROWO',
    'World':'World'}


def create_macro_df(idf: pyam.IamDataFrame):
    """Converts data at country level to macro-region data"""

    # Dataset preparation
    df_country = idf.timeseries()
    df_country.index.names = ['model', 'scenario', 'country', 'variable', 'unit']
    
    # Label each country based on what region it's part of
    df_country = df_country.pix.semijoin(REMIND_RMAP.index, how="left")

    # Kick out countries which don't have a region mapping               
    df_country=df_country.loc[~isin(region=np.nan)]
    
    # # Groupby regions and sum (much faster than doing in pyam)
    df_country = pyam.IamDataFrame(
        df_country
        .groupby(['model','scenario','variable','unit','region'])
        .sum())
    
    df_region = pyam.IamDataFrame(df_country)
       
    # Drop 2023 data as it is currently missing quite a few countries (and so the regional totals are off)
    df_region.filter(year=2023,keep=False,inplace=True)

    # Rename regions
    df_region = (df_region
                 .rename(region=region_map)
                 .filter(region=['SSA','CHA','NEU','CAZ','EUR','IND','LAM','MEA','USA','JPN','REF','OAS'])
                )

    # Add world data
    for var in df_region.variable:
        df_region.aggregate_region(
            var,
            'World',
            append=True)

    return pyam.IamDataFrame(df_region.timeseries())

In [149]:
dscale_results_agg = create_macro_df(dscale_results)

In [150]:
hist_agg = create_macro_df(hist_data)
hist_agg.filter(year=[2023,2024],keep=False,inplace=True) 
#Drop 2023/2024 data as underlying country data is regionally incomplete and so you get aggregation errors

## Regional consistencies

In [170]:
# ── dscale_results_agg  vs  input_data ───────────────────────────────────────
# Verify that re-aggregating country-level results recovers the original
# regional input data, for every variable present in both.

common_vars    = sorted(set(input_data.filter(variable = ["Emissions*", "Primary*", "Secondary*", "Final*"]).variable) & set(dscale_results_agg.filter(variable = ["Emissions*", "Primary*", "Secondary*", "Final*"]).variable))
common_regions = sorted(set(input_data.filter(region = ["World", "CHA", "JPN", "USA", "IND"], keep=False).region)   & set(dscale_results_agg.filter(region = ["World", "CHA", "JPN", "USA", "IND"], keep=False).region))
common_regions = sorted(set(input_data.region) & set(dscale_results_agg.region))


print(f"Common: {len(common_vars)} variables  ×  {len(common_regions)} regions")
print(f"Regions: {common_regions}\n")

# Extract timeseries; drop model/scenario to align on (region, variable, unit)
ref_ts = input_data.filter(variable=common_vars, region=common_regions).timeseries()
dsc_ts = dscale_results_agg.filter(variable=common_vars, region=common_regions).timeseries()

ref_ts.index = ref_ts.index.droplevel(['model', 'scenario'])
dsc_ts.index = dsc_ts.index.droplevel(['model', 'scenario'])

# Align on shared years and index entries
common_years   = sorted(set(ref_ts.columns) & set(dsc_ts.columns))
ref_ts, dsc_ts = ref_ts[common_years].align(dsc_ts[common_years], join='inner')

# Max absolute difference per (region, variable)
max_abs  = (ref_ts - dsc_ts).abs().max(axis=1)
# max_abs  = (ref_ts - dsc_ts).max(axis=1)
summary  = max_abs.reset_index()
summary.columns = ['region', 'variable', 'unit', 'max_abs_diff']
summary  = summary.sort_values('max_abs_diff', ascending=False).reset_index(drop=True)

TOL_AGG  = 0.01  # EJ/yr — adjust as needed
discrepancies = summary[summary['max_abs_diff'] > TOL_AGG]

print(f"Variable × region pairs checked : {len(summary)}")
print(f"Tolerance                        : {TOL_AGG} (units in data, e.g. EJ/yr)")
if discrepancies.empty:
    print("✓  All values match within tolerance")
else:
    print(f"✗  {len(discrepancies)} pairs exceed tolerance\n")
    display(discrepancies)

Common: 242 variables  ×  13 regions
Regions: ['CAZ', 'CHA', 'EUR', 'IND', 'JPN', 'LAM', 'MEA', 'NEU', 'OAS', 'REF', 'SSA', 'USA', 'World']

Variable × region pairs checked : 1830
Tolerance                        : 0.01 (units in data, e.g. EJ/yr)
✗  764 pairs exceed tolerance



,region,variable,unit,max_abs_diff
0,World,Emissions|CO2,Mt CO2/yr,4984.219209
1,World,Emissions|CO2|Energy|Demand|Transportation,Mt CO2/yr,4709.860107
2,CHA,Emissions|CO2,Mt CO2/yr,2033.050500
3,REF,Emissions|CO2,Mt CO2/yr,1729.473100
4,World,Emissions|CO2|Energy,Mt CO2/yr,1720.684408
...,...,...,...,...
759,IND,Secondary Energy|Electricity|Nuclear,EJ/yr,0.011380
760,CAZ,Final Energy|Residential and Commercial|Electr...,EJ/yr,0.011137
761,MEA,Secondary Energy|Electricity|Coal,EJ/yr,0.010945
762,CHA,Primary Energy|Fossil|w/ CCS,EJ/yr,0.010358


In [171]:
discrepancies.loc[discrepancies["variable"] == "Final Energy"]

,region,variable,unit,max_abs_diff
61,World,Final Energy,EJ/yr,47.363279
132,CHA,Final Energy,EJ/yr,13.178960
166,USA,Final Energy,EJ/yr,8.004965
180,IND,Final Energy,EJ/yr,6.679337
198,LAM,Final Energy,EJ/yr,5.257291
235,SSA,Final Energy,EJ/yr,3.784978
245,OAS,Final Energy,EJ/yr,3.482237
261,MEA,Final Energy,EJ/yr,2.889086
282,EUR,Final Energy,EJ/yr,2.386213
314,REF,Final Energy,EJ/yr,1.785520


In [168]:
discrepancies.loc[discrepancies["variable"] == "Final Energy|Transportation"]

,region,variable,unit,max_abs_diff
80,LAM,Final Energy|Transportation,EJ/yr,2.396519
96,EUR,Final Energy|Transportation,EJ/yr,1.743450
103,MEA,Final Energy|Transportation,EJ/yr,1.496391
122,OAS,Final Energy|Transportation,EJ/yr,1.218620
166,SSA,Final Energy|Transportation,EJ/yr,0.835374
214,REF,Final Energy|Transportation,EJ/yr,0.350914
232,NEU,Final Energy|Transportation,EJ/yr,0.286035
247,CAZ,Final Energy|Transportation,EJ/yr,0.223493


## Final Energy consistencies

In [161]:
_ts   = dscale_results.filter(region = ["CHN", "IND", "USA", "JPN"], keep=False).timeseries()
_vars = set(_ts.index.get_level_values('variable'))
TOL   = 0.01   # EJ/yr — absolute tolerance for all FE checks

In [162]:
# ── Pre-load country-level timeseries once (used by all checks below) ────────
# _ts   = dscale_results.filter(region = ["CHN", "IND", "USA", "JPN"], keep=False).timeseries()
_ts   = dscale_results.filter(region = ["CHN", "IND", "USA", "JPN"], keep=False).timeseries()
_vars = set(_ts.index.get_level_values('variable'))
TOL   = 0.01   # EJ/yr — absolute tolerance for all FE checks


def check_sum(agg_var, comp_vars):
    """Sum comp_vars and compare to agg_var across all countries.
    Returns a DataFrame of absolute differences: rows = countries, cols = years.
    Prints a warning for any comp_vars missing from the data.
    """
    found   = [v for v in comp_vars if v in _vars]
    missing = [v for v in comp_vars if v not in _vars]
    if missing:
        print(f"    ⚠ not in data: {missing}")
    if agg_var not in _vars or not found:
        return pd.DataFrame()

    agg  = _ts.loc[_ts.index.isin([agg_var], level='variable')].droplevel('variable')
    ssum = (_ts.loc[_ts.index.isin(found, level='variable')]
               .groupby(level=['model', 'scenario', 'region', 'unit']).sum())

    agg, ssum = agg.align(ssum, join='inner')
    return (agg - ssum).abs()                          # full country × year grid


def summarise(checks, title):
    """Run check_sum for every {agg_var: [comp_vars]} entry; display a summary table."""
    rows = []
    for agg_var, comp_vars in checks.items():
        diffs = check_sum(agg_var, comp_vars)          # DataFrame: countries × years
        if diffs.empty:
            rows.append(dict(check=agg_var, countries=0, n_fail_countries=0,
                             n_fail_years=0, max_diff=np.nan, worst="N/A"))
            continue

        max_per_country = diffs.max(axis=1)            # worst year per country
        n_fail_countries = int((max_per_country > TOL).sum())
        n_fail_years     = int((diffs > TOL).sum().sum())   # total failing country×year cells

        rows.append(dict(
            check            = agg_var,
            countries        = len(diffs),
            n_fail_countries = n_fail_countries,
            n_fail_years     = n_fail_years,
            max_diff         = round(float(max_per_country.max()), 6),
            worst            = max_per_country.idxmax()[2] if n_fail_countries > 0 else "–",
        ))

    out = pd.DataFrame(rows)
    out.insert(0, '', out['n_fail_countries'].map(lambda x: '✓' if x == 0 else '✗'))
    print(f"\n{'─'*70}\n  {title}\n{'─'*70}")
    # display(out)
    return out




### Fuel are consistent with their carriers within each sector

In [98]:
# ── Fuel → Carrier definitions (only carriers that have sub-fuel splits) ─────
FUEL_TO_CARRIER = {
    # Industry
    "Final Energy|Industry|Gases" : ["Final Energy|Industry|Gases|Biomass",
                                     "Final Energy|Industry|Gases|Natural Gas",
                                     "Final Energy|Industry|Gases|Hydrogen synfuel",
                                     "Final Energy|Industry|Gases|Electricity"],
    "Final Energy|Industry|Liquids": ["Final Energy|Industry|Liquids|Biomass",
                                      "Final Energy|Industry|Liquids|Oil",
                                      "Final Energy|Industry|Liquids|Electricity"],
    "Final Energy|Industry|Solids" : ["Final Energy|Industry|Solids|Biomass",
                                      "Final Energy|Industry|Solids|Coal"],
    # Residential & Commercial
    "Final Energy|Residential and Commercial|Gases" : ["Final Energy|Residential and Commercial|Gases|Biomass",
                                                        "Final Energy|Residential and Commercial|Gases|Natural Gas",
                                                        "Final Energy|Residential and Commercial|Gases|Electricity"],
    "Final Energy|Residential and Commercial|Liquids": ["Final Energy|Residential and Commercial|Liquids|Biomass",
                                                         "Final Energy|Residential and Commercial|Liquids|Oil",
                                                         "Final Energy|Residential and Commercial|Liquids|Electricity"],
    "Final Energy|Residential and Commercial|Solids" : ["Final Energy|Residential and Commercial|Solids|Biomass",
                                                         "Final Energy|Residential and Commercial|Solids|Coal"],
    # Transportation
    "Final Energy|Transportation|Gases" : ["Final Energy|Transportation|Gases|Biomass",
                                            "Final Energy|Transportation|Gases|Natural Gas",
                                            "Final Energy|Transportation|Gases|Electricity"],
    "Final Energy|Transportation|Liquids": ["Final Energy|Transportation|Liquids|Biomass",
                                             "Final Energy|Transportation|Liquids|Oil",
                                             "Final Energy|Transportation|Liquids|Coal",
                                             "Final Energy|Transportation|Liquids|Electricity"],
}

summarise(FUEL_TO_CARRIER, "Fuels  →  Carrier  (per sector)")

    ⚠ not in data: ['Final Energy|Industry|Gases|Hydrogen synfuel']

──────────────────────────────────────────────────────────────────────
  Fuels  →  Carrier  (per sector)
──────────────────────────────────────────────────────────────────────


,,check,countries,n_fail_countries,n_fail_years,max_diff,worst
0,✗,Final Energy|Industry|Gases,31,12,52,0.096460,PRK
1,✗,Final Energy|Industry|Liquids,31,6,16,0.063024,KOR
2,✗,Final Energy|Industry|Solids,31,11,71,0.133100,PAK
3,✗,Final Energy|Residential and Commercial|Gases,31,6,14,0.061308,KOR
4,✗,Final Energy|Residential and Commercial|Liquids,31,4,15,0.085778,KOR
5,✗,Final Energy|Residential and Commercial|Solids,31,4,12,0.144100,LKA
6,✗,Final Energy|Transportation|Gases,31,3,5,0.027181,LKA
7,✗,Final Energy|Transportation|Liquids,31,7,22,0.095189,LKA


### Carriers are consistent with the total sector 

In [50]:
# ── Carriers  →  Sector total ─────────────────────────────────────────────────
# Note: REMIND uses "Statistical Difference" adjustment variables in some
# sectors.  If a sector has one, expect a small residual here.

CARRIERS_IN_SECTOR = {
    "Final Energy|Industry": [
        "Final Energy|Industry|Electricity",
        "Final Energy|Industry|Gases",
        "Final Energy|Industry|Heat",
        "Final Energy|Industry|Hydrogen",
        "Final Energy|Industry|Liquids",
        "Final Energy|Industry|Solids",
    ],
    "Final Energy|Residential and Commercial": [
        "Final Energy|Residential and Commercial|Electricity",
        "Final Energy|Residential and Commercial|Gases",
        "Final Energy|Residential and Commercial|Heat",
        "Final Energy|Residential and Commercial|Hydrogen",
        "Final Energy|Residential and Commercial|Liquids",
        "Final Energy|Residential and Commercial|Solids",
    ],
    "Final Energy|Transportation": [
        "Final Energy|Transportation|Electricity",
        "Final Energy|Transportation|Gases",
        "Final Energy|Transportation|Hydrogen",
        "Final Energy|Transportation|Liquids",
        # no Heat / Solids in Transportation
    ],
    "Final Energy|Other Sector": [
        "Final Energy|Other Sector|Electricity",
        "Final Energy|Other Sector|Gases",
        "Final Energy|Other Sector|Heat",
        "Final Energy|Other Sector|Hydrogen",
        "Final Energy|Other Sector|Liquids",
        # no Solids in Other Sector
    ],
}

summarise(CARRIERS_IN_SECTOR, "Carriers  →  Sector total")

    ⚠ not in data: ['Final Energy|Other Sector|Electricity', 'Final Energy|Other Sector|Gases', 'Final Energy|Other Sector|Heat', 'Final Energy|Other Sector|Hydrogen', 'Final Energy|Other Sector|Liquids']

──────────────────────────────────────────────────────────────────────
  Carriers  →  Sector total
──────────────────────────────────────────────────────────────────────


,,check,countries,n_fail_countries,n_fail_years,max_diff,worst
0,✗,Final Energy|Industry,189,116,1079,2.2561,EGY
1,✗,Final Energy|Residential and Commercial,189,101,853,0.7937,VNM
2,✗,Final Energy|Transportation,189,98,803,2.1455,KOR
3,✓,Final Energy|Other Sector,0,0,0,NaN,N/A


### Carriers in sector consistent with FE|Carriers

In [134]:
# ── Sum of sector-level carriers  →  top-level FE|Carrier ────────────────────
# e.g. FE|Electricity = Industry|Elec + R&C|Elec + Transport|Elec + Other|Elec

CARRIER_ACROSS_SECTORS = {
    "Final Energy|Electricity": [
        "Final Energy|Industry|Electricity",
        "Final Energy|Residential and Commercial|Electricity",
        "Final Energy|Transportation|Electricity",
        "Final Energy|Other Sector|Electricity",
    ],
    "Final Energy|Gases": [
        "Final Energy|Industry|Gases",
        "Final Energy|Residential and Commercial|Gases",
        "Final Energy|Transportation|Gases",
        "Final Energy|Other Sector|Gases",
    ],
    "Final Energy|Heat": [
        "Final Energy|Industry|Heat",
        "Final Energy|Residential and Commercial|Heat",
        "Final Energy|Other Sector|Heat",
        # Transportation has no Heat
    ],
    "Final Energy|Hydrogen": [
        "Final Energy|Industry|Hydrogen",
        "Final Energy|Residential and Commercial|Hydrogen",
        "Final Energy|Transportation|Hydrogen",
        "Final Energy|Other Sector|Hydrogen",
    ],
    "Final Energy|Liquids": [
        "Final Energy|Industry|Liquids",
        "Final Energy|Residential and Commercial|Liquids",
        "Final Energy|Transportation|Liquids",
        "Final Energy|Other Sector|Liquids",
    ],
    "Final Energy|Solids": [
        "Final Energy|Industry|Solids",
        "Final Energy|Residential and Commercial|Solids",
        # Transportation and Other Sector have no Solids
    ],
}

summarise(CARRIER_ACROSS_SECTORS, "Sum(Sector|Carrier) across sectors  →  FE|Carrier")

    ⚠ not in data: ['Final Energy|Other Sector|Electricity']
    ⚠ not in data: ['Final Energy|Other Sector|Gases']
    ⚠ not in data: ['Final Energy|Other Sector|Heat']
    ⚠ not in data: ['Final Energy|Other Sector|Hydrogen']
    ⚠ not in data: ['Final Energy|Other Sector|Liquids']

──────────────────────────────────────────────────────────────────────
  Sum(Sector|Carrier) across sectors  →  FE|Carrier
──────────────────────────────────────────────────────────────────────


,,check,countries,n_fail_countries,n_fail_years,max_diff,worst
0,✗,Final Energy|Electricity,31,15,144,2.106700,KOR
1,✗,Final Energy|Gases,31,12,87,0.434885,PAK
2,✗,Final Energy|Heat,31,5,32,0.411000,IDN
3,✓,Final Energy|Hydrogen,0,0,0,NaN,N/A
4,✗,Final Energy|Liquids,31,17,189,1.023094,IDN
5,✗,Final Energy|Solids,31,12,69,0.203500,PAK


### Sum(Sector) == Final Energy

In [ ]:
# ── Sum of sectors  →  Final Energy ───────────────────────────────────────────

SECTORS = {
    "Final Energy|excluding DAC": [
        "Final Energy|Industry",
        "Final Energy|Residential and Commercial",
        "Final Energy|Transportation",
        "Final Energy|Other Sector",
    ],
}

summarise(SECTORS, "Sum(Sectors)  →  Final Energy")

    ⚠ not in data: ['Final Energy|Other Sector']

──────────────────────────────────────────────────────────────────────
  Sum(Sectors)  →  Final Energy
──────────────────────────────────────────────────────────────────────


,,check,countries,n_fail_countries,n_fail_years,max_diff,worst
0,✗,Final Energy,31,17,291,2.6866,PAK


### Sum(Carriers) == Final Energy

In [129]:
# ── Sum of carriers  →  Final Energy ──────────────────────────────────────────

CARRIERS = {
    "Final Energy": [
        "Final Energy|Electricity",
        "Final Energy|Gases",
        "Final Energy|Heat",
        "Final Energy|Hydrogen",
        "Final Energy|Liquids",
        "Final Energy|Solids",
    ],
}

summarise(CARRIERS, "Sum(Carriers)  →  Final Energy")

    ⚠ not in data: ['Final Energy|Hydrogen']

──────────────────────────────────────────────────────────────────────
  Sum(Carriers)  →  Final Energy
──────────────────────────────────────────────────────────────────────


,,check,countries,n_fail_countries,n_fail_years,max_diff,worst
0,✗,Final Energy,31,21,297,7.8428,IDN


# Final Energy|Electricity ≤ Secondary Energy|Electricity

In [163]:
# ── FE|Electricity must not exceed SE|Electricity ─────────────────────────────
# Electricity consumption (FE) can never be larger than production (SE).

fe_var = "Final Energy|Electricity"
se_var = "Secondary Energy|Electricity"

fe_ts = _ts.loc[_ts.index.isin([fe_var], level='variable')].droplevel('variable')
se_ts = _ts.loc[_ts.index.isin([se_var], level='variable')].droplevel('variable')

fe_ts, se_ts = fe_ts.align(se_ts, join='inner')

# Excess = FE − SE;  only positive values are violations
excess = fe_ts - se_ts
violations = excess > TOL                                  # boolean mask

n_violations = int(violations.sum().sum())                 # total country × year cells
n_countries  = int(violations.any(axis=1).sum())           # countries with ≥1 bad year

if n_violations == 0:
    print("✓  FE|Electricity ≤ SE|Electricity everywhere")
else:
    print(f"✗  {n_violations} country × year violations  ({n_countries} countries)\n")
    print(f"   Max excess : {excess.max().max():.6f}\n")

    # Worst offenders: max excess per country, descending
    worst = (excess.max(axis=1)
             .pipe(lambda s: s[s > TOL])
             .sort_values(ascending=False)
             .rename('max_excess'))
    worst.index = worst.index.droplevel(['model', 'scenario', 'unit'])
    display(worst.reset_index().head(20))

✗  331 country × year violations  (68 countries)

   Max excess : 8.805900



,region,max_excess
0,IDN,8.8059
1,BGD,1.6086
2,PAK,1.0013
3,VEN,0.7378
4,NER,0.5194
5,PER,0.4255
6,COL,0.4042
7,KWT,0.3932
8,UGA,0.3247
9,LKA,0.3068


# Primary Energy ≥ Final Energy (fuel supply)

Each fossil fuel consumed in final energy must be covered by primary energy supply.
PE includes net trade (imports), so PE ≥ FE should hold at the **regional** level.

At the **country** level, however, REMIND does not track intra-regional fuel trade: PE and FE are downscaled with different geographic shares, so oil- and gas-importing countries routinely show FE > PE. This is a structural feature of the downscaling, not a bug.

* **Coal**: no violations expected (coal trade within REMIND regions is small).
* **Oil / Gas**: country-level violations are expected and should not be "fixed" by clipping — doing so would artificially cut consumption in importing countries.

In [92]:
# ── PE ≥ FE fuel-level checks ─────────────────────────────────────────────────
# Positive excess  =  more of that fuel consumed in FE than supplied in PE.
# For Oil and Gas this is expected for intra-regional importing countries.

PE_GE_FE = {
    "Coal":        ("Primary Energy|Coal",  ["Final Energy|Solids|Coal"]),
    "Oil":         ("Primary Energy|Oil",   ["Final Energy|Industry|Liquids|Oil",
                                             "Final Energy|Residential and Commercial|Liquids|Oil",
                                             "Final Energy|Transportation|Liquids|Oil"]),
    "Natural Gas": ("Primary Energy|Gas",   ["Final Energy|Industry|Gases|Natural Gas",
                                             "Final Energy|Residential and Commercial|Gases|Natural Gas",
                                             "Final Energy|Transportation|Gases|Natural Gas"]),
}

rows = []
for fuel, (pe_var, fe_vars) in PE_GE_FE.items():
    found_fe = [v for v in fe_vars if v in _vars]
    if pe_var not in _vars or not found_fe:
        continue

    pe_ts = _ts.loc[_ts.index.isin([pe_var], level='variable')].droplevel('variable')
    fe_ts = (_ts.loc[_ts.index.isin(found_fe, level='variable')]
                .groupby(level=['model', 'scenario', 'region', 'unit']).sum())
    pe_ts, fe_ts = pe_ts.align(fe_ts, join='inner')

    excess           = fe_ts - pe_ts                       # >0  →  FE > PE
    max_per_country  = excess.max(axis=1)
    n_fail_countries = int((max_per_country > TOL).sum())
    n_fail_years     = int((excess > TOL).sum().sum())

    rows.append(dict(
        fuel             = fuel,
        PE               = pe_var,
        n_fail_countries = n_fail_countries,
        n_fail_years     = n_fail_years,
        max_excess       = round(float(max_per_country.max()), 4),
        worst            = max_per_country.idxmax()[2] if n_fail_countries > 0 else "–",
    ))

out = pd.DataFrame(rows)
out.insert(0, '', out['n_fail_countries'].map(lambda x: '✓' if x == 0 else '✗'))
print(f"\n{'─'*70}\n  Primary Energy ≥ Final Energy (fuel supply)\n{'─'*70}")
display(out)


──────────────────────────────────────────────────────────────────────
  Primary Energy ≥ Final Energy (fuel supply)
──────────────────────────────────────────────────────────────────────


,,fuel,PE,n_fail_countries,n_fail_years,max_excess,worst
0,✗,Oil,Primary Energy|Oil,6,26,0.6519,KOR
1,✗,Natural Gas,Primary Energy|Gas,7,45,0.1233,PHL


Why no code fix was applied for oil/gas:

The oil and gas violations are not a bug — they're an expected consequence of the downscaling methodology. I verified this by checking the REMIND regional input: at the regional level, PE|Oil > FE|Oil everywhere (the energy balance is correct). The country-level violations exist because REMIND doesn't track intra-regional fuel trade. When PE and FE are distributed to countries using different shares (PE via production shares, FE via consumption shares), importing countries end up with FE > PE. Clipping would be incorrect — it would artificially cut consumption in those countries. Coal has no violations because intra-regional coal trade is negligible compared to oil/gas.

The violations pre-exist before step5e (92 countries in Step5d for oil), so this isn't a harmonization artifact either — it's structural to the downscaling approach.